In [1]:
import xml.etree.ElementTree as ET

In [3]:
tree = ET.parse('../_data/MDB_STAMMDATEN.XML')
root = tree.getroot()
bt_list = []

for mdb in root.findall('MDB'):
    processed_names = set()
    
     # Names can change due to marriage etc. -> iterate over all listed names for the MDB
    for name_element in mdb.findall('.//NAME'):
        last_name = name_element.find('NACHNAME').text
        first_name = name_element.find('VORNAME').text
        full_name = f"{first_name} {last_name}"
        election_period =  mdb.find('.//WP').text
        
        # We are only want to get multiple entries where first or last name changed, not e.g. Title "Dr." etc.
        if full_name not in processed_names:

            '''
                Edge case, Abbreviated second first names:
                    e.g. Lutz G. Stavenhagen is sometimes referred to by Lutz Stavenhagen in the protocols.
                    This is why we need to put both version "Lutz Stavenhagen" and "Lutz G. Stavenhagen" into the list.

                Edge case of the edge case: Since BT-Period 18, sometimes the data in the xml file reflects if a 
                    MDB changed how he is referred to in the protocols. This would lead to double entries for Albert (H.) Weiler and Tobias (B.) Bacherle.
                    This is why we have to check if the Bundestagsperiode is smaller than 18 before adding the abbreviated part to the list 
                    since beginning with bt-period 18, the "double entry" will automatically be created through the findall('.//NAME')
            '''
            if first_name[-1] == '.' and int(election_period) <18:
                dict_entry_case_abbreviated = {
                'last_name': last_name,
                'first_name': first_name,
                'anrede': name_element.find('ANREDE_TITEL').text,
                'party': mdb.find('.//PARTEI_KURZ').text,
                'election_period': election_period
                }
                bt_list.append(dict_entry_case_abbreviated)
                first_name = first_name[:-3] # get rid of abbreviation part

            dict_entry = {
                'last_name': last_name,
                'first_name': first_name,
                'anrede': name_element.find('ANREDE_TITEL').text,
                'party': mdb.find('.//PARTEI_KURZ').text,
                'election_period': election_period
            }
            if dict_entry['party'] in ['BÜNDNIS 90/DIE GRÜNEN', 'DIE GRÜNEN/BÜNDNIS 90']:
                dict_entry['party'] = "GRÜNE"
            
            bt_list.append(dict_entry)
            processed_names.add(full_name) 

Das enthält mehr Daten, als wir eigentlich brauchen, deswegen hier eine vereinfachte Form, ohne zusätzliche Infos:

In [4]:
bt_tuples = [(f"{adbt['first_name']} {adbt['last_name']}", adbt['party'], adbt['election_period']) for adbt in bt_list]
bt_tuples = sorted(bt_tuples, key = lambda x: int(x[2]), reverse = True)
bt_tuples

[('Dieter Janecek', 'GRÜNE', '20'),
 ('Carsten Brodesser', 'CDU', '20'),
 ('Kirsten Kappert-Gonther', 'GRÜNE', '20'),
 ('Bettina Margarethe Wiesmann', 'CDU', '20'),
 ('Sanae Abdi', 'SPD', '20'),
 ('Valentin Abel', 'FDP', '20'),
 ('Knut Abraham', 'CDU', '20'),
 ('Katja Adler', 'FDP', '20'),
 ('Stephanie Aeffner', 'GRÜNE', '20'),
 ('Adis Ahmetovic', 'SPD', '20'),
 ('Reem Alabali-Radovan', 'SPD', '20'),
 ('Ali Al-Dailami', 'Plos', '20'),
 ('Muhanad Al-Halak', 'FDP', '20'),
 ('Dagmar Andres', 'SPD', '20'),
 ('Johannes Arlt', 'SPD', '20'),
 ('Andreas Audretsch', 'GRÜNE', '20'),
 ('Maik Außendorf', 'GRÜNE', '20'),
 ('Tobias Bacherle', 'GRÜNE', '20'),
 ('Tobias B. Bacherle', 'GRÜNE', '20'),
 ('Carolin Bachmann', 'AfD', '20'),
 ('Daniel Baldy', 'SPD', '20'),
 ('Felix Banaszak', 'GRÜNE', '20'),
 ('Karl Bär', 'GRÜNE', '20'),
 ('Christina Baum', 'AfD', '20'),
 ('Katharina Beck', 'GRÜNE', '20'),
 ('Roger Beckamp', 'AfD', '20'),
 ('Holger Becker', 'SPD', '20'),
 ('Lukas Benner', 'GRÜNE', '20'),
 ('

In [5]:
def find_duplicates_by_name(list_of_tuples):
    name_map = {}
    duplicates = []
    
    for tup in list_of_tuples:
        name = tup[0]
        if name in name_map:
            name_map[name].append(tup)
            if len(name_map[name]) == 2:
                duplicates.extend(name_map[name])
        else:
            name_map[name] = [tup]
    
    return duplicates

find_duplicates_by_name(bt_tuples)

[('Peter Friedrich', 'SPD', '16'),
 ('Peter Friedrich', 'SPD', '14'),
 ('Dagmar Schmidt', 'SPD', '18'),
 ('Dagmar Schmidt', 'SPD', '13'),
 ('Frank Schmidt', 'SPD', '14'),
 ('Frank Schmidt', 'CDU', '11'),
 ('Michael Müller', 'SPD', '20'),
 ('Michael Müller', 'SPD', '10'),
 ('Christian Schmidt', 'CSU', '12'),
 ('Christian Schmidt', 'GRÜNE', '10'),
 ('Karl Lamers', 'CDU', '13'),
 ('Karl Lamers', 'CDU', '9'),
 ('Rudolf Müller', 'SPD', '7'),
 ('Rudolf Müller', 'CDU', '6'),
 ('Manfred Schmidt', 'CDU', '7'),
 ('Manfred Schmidt', 'SPD', '6'),
 ('Alois Rainer', 'CSU', '18'),
 ('Alois Rainer', 'CSU', '5'),
 ('Günter Klein', 'CDU', '12'),
 ('Günter Klein', 'SPD', '4'),
 ('Karl Müller', 'SPD', '3'),
 ('Karl Müller', 'DP', '2'),
 ('Harald Koch', 'DIE LINKE', '17'),
 ('Harald Koch', 'SPD', '1'),
 ('Heinrich Müller', 'SPD', '4'),
 ('Heinrich Müller', 'SPD', '1'),
 ('Wilhelm Schmidt', 'SPD', '11'),
 ('Wilhelm Schmidt', 'WAV', '1'),
 ('Gerhard Schröder', 'SPD', '9'),
 ('Gerhard Schröder', 'CDU', '1'),


#### Zusammenfassend

Relativ wenige doppelte Einträge. Immerhin sind über 4000 unterschiedliche Bundestagsabgeordnete Teil der Liste.  
Die Liste wird sowieso im Parsing-Teil nur genutzt, um die Parteizugehörigkeit von Ministern, Bundeskanzlern und Staatssekretären nachzuschlagen.  
Bei normalen Abgeordneten steht die Parteizugehörigkeit mit im Text. Deswegen sollten diese doppelten Namen kaum Einfluss auf die Endergebnisse unserer Analysen haben (wenn überhaupt).

In [6]:
len(bt_tuples)

4497

In [7]:
import re
def find_party_by_name(name, tuple_list):
    tuple_list_sorted = sorted(tuple_list, key = lambda x: x[2], reverse = True) # we focus on 
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list_sorted:
        if name in item[0]:
            if item[1] != 'CSU' and item[1] != 'CDU':
                return item[1]
            else:
                return 'CDU/CSU'  
    return "<unknown>"

print(find_party_by_name ('Dr. Norbert Blüm', bt_tuples))
print(find_party_by_name ('Ursula Seiler-Albring', bt_tuples))

CDU/CSU
FDP


Einige Staatssekretäre und Minister waren vor ihrem Amt nicht im Bundestag, kommen also auch in der Liste nicht vor. 
Die Wichtigsten (am häufigsten vorkommenden) habe ich noch händisch meistens über Wikipedia herausgesucht, damit deren Reden auch der richtigen Partei zugeordnet werden können. 

In [8]:
bt_tuples.append(("Boris Pistorius", "SPD", 20))
bt_tuples.append(("Nancy Faeser", "SPD", 20))
bt_tuples.append(("Klaus-Dieter Fritsche", "CDU/CSU", 19))
bt_tuples.append(("Aydan Özoguz", "SPD", 19)) # Member of BT but the ğ accent is not always present in the protocols, so we append her manually
bt_tuples.append(("Johanna Wanka", "CDU/CSU", 18))
bt_tuples.append(("Philipp Rösler", "FDP", 17))
bt_tuples.append(("Hans-Jürgen Beerfeltz", "FDP", 17))
bt_tuples.append(("Erich Stather", "SPD",16))
bt_tuples.append(("Wolfgang Clement", "SPD", 14))
bt_tuples.append(("Christina Weiss", "fraktionslos", 14))
bt_tuples.append(("Julian Nida-Rümelin", "SPD", 14))
bt_tuples.append(("Michael Naumann", "SPD", 14))
bt_tuples.append(("Werner Tegtmeier", "SPD", 13))
bt_tuples.append(("Jürgen Stark", "fraktionslos", 13))
bt_tuples.append(("Hans-Friedrich von Ploetz", "fraktionslos", 13))
bt_tuples.append(("Baldur Wagner", "CDU/CSU", 12))
bt_tuples.append(("Karl Jung", "fraktionslos", 12))
bt_tuples.append(("Franz-Josef Feiter", "fraktionslos", 12))
bt_tuples.append(("Manfred Overhaus", "fraktionslos", 12))
bt_tuples.append(("Wighard Härdtl", "CDU/CSU", 12))
bt_tuples.append(("Wilhelm Knittel", "CDU/CSU", 12))
bt_tuples.append(("Clemens Stroetmann", "CDU/CSU", 12))
bt_tuples.append(("Franz Kroppenstedt", "fraktionslos", 12))
bt_tuples.append(("Hans-Joachim Fuchtel", "CDU/CSU", 12))
bt_tuples.append(("Frerich Görts", "CDU/CSU", 12))

Speichern in einer CSV-Datei, damit wir im nächsten Notebook leichten Zugriff darauf haben.

In [10]:
import csv

with open('../_data/abgeordnete.csv','w', encoding='utf8', newline='') as out:
    csv_out=csv.writer(out)
    csv_out.writerow(['name','party', 'BT-Period'])
    csv_out.writerows(bt_tuples) 
